# iTransformer -- ETTh1 Training

Reference: Liu et al., *iTransformer: Inverted Transformers Are Effective for Time Series Forecasting*,
ICLR 2024 Spotlight. https://arxiv.org/abs/2310.06625

Trains iTransformer at four forecast horizons (96, 192, 336, 720) on ETTh1 (multivariate).
Target: averaged MSE ~0.454, MAE ~0.447 (Liu et al., Table 1).

All model and dataset code is inlined so this notebook runs without a local package install.

In [2]:
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cuda


## Dataset

In [3]:
class ETTh1Dataset(Dataset):
    """ETTh1 multivariate dataset with chronological train/val/test split.

    Split follows the protocol used in PatchTST and iTransformer papers:
        train: first 8640 rows  (12 months)
        val:   next  2880 rows  (4 months)
        test:  final 2880 rows  (4 months)

    Normalization: z-score per channel, fit on train split only.
    """

    SPLIT_SIZES = {'train': 8640, 'val': 2880, 'test': 2880}
    TARGET_COLS = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT']

    def __init__(self, csv_path: str, split: str, seq_len: int, pred_len: int) -> None:
        if split not in self.SPLIT_SIZES:
            raise ValueError(f"split must be one of {list(self.SPLIT_SIZES)}, got '{split}'.")

        df = pd.read_csv(csv_path)[self.TARGET_COLS].values.astype(np.float32)

        train_end = self.SPLIT_SIZES['train']
        val_end = train_end + self.SPLIT_SIZES['val']

        # Fit scaler on train only -- no leakage.
        train_data = df[:train_end]
        self._mean = train_data.mean(axis=0)
        self._std = train_data.std(axis=0)
        self._std = np.where(self._std == 0, 1.0, self._std)

        data = (df - self._mean) / self._std

        if split == 'train':
            self._data = data[:train_end]
        elif split == 'val':
            self._data = data[train_end:val_end]
        else:
            self._data = data[val_end:]

        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self) -> int:
        return len(self._data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx: int):
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)

## Model

In [4]:
class EncoderLayer(nn.Module):
    """Pre-norm transformer encoder layer."""

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x


class TransformerEncoder(nn.Module):
    """Stack of EncoderLayer blocks."""

    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)


class VariateEmbedding(nn.Module):
    """Project each variate's full history to a d_model token."""

    def __init__(self, seq_len: int, d_model: int) -> None:
        super().__init__()
        self.projection = nn.Linear(seq_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (B, seq_len, C) -> (B, C, seq_len) -> (B, C, d_model)
        return self.projection(x.transpose(1, 2))


class ForecastHead(nn.Module):
    """Per-variate projection from d_model to pred_len."""

    def __init__(self, d_model: int, pred_len: int) -> None:
        super().__init__()
        self.projection = nn.Linear(d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (B, C, d_model) -> (B, C, pred_len) -> (B, pred_len, C)
        return self.projection(x).transpose(1, 2)


class iTransformer(nn.Module):
    """iTransformer: inverted tokenization for multivariate forecasting."""

    def __init__(self, seq_len: int, pred_len: int, d_model: int = 512, num_heads: int = 8, num_layers: int = 3, dropout: float = 0.1) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f'd_model ({d_model}) must be divisible by num_heads ({num_heads}).')
        self.embedding = VariateEmbedding(seq_len=seq_len, d_model=d_model)
        self.encoder = TransformerEncoder(d_model=d_model, num_heads=num_heads, num_layers=num_layers, dropout=dropout)
        self.head = ForecastHead(d_model=d_model, pred_len=pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (B, seq_len, C) -> (B, C, d_model) -> (B, C, d_model) -> (B, pred_len, C)
        return self.head(self.encoder(self.embedding(x)))

## Training Infrastructure

In [5]:
class EarlyStopping:
    """Stop training when validation MSE has not improved for `patience` epochs."""

    def __init__(self, patience: int = 10, checkpoint_path: str = 'best_model.pt') -> None:
        self.patience = patience
        self.checkpoint_path = checkpoint_path
        self.best_val_mse = float('inf')
        self.counter = 0
        self.best_epoch = 0

    def step(self, val_mse: float, model: nn.Module, epoch: int) -> bool:
        """Return True if training should stop."""
        if val_mse < self.best_val_mse:
            self.best_val_mse = val_mse
            self.counter = 0
            self.best_epoch = epoch
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> tuple[float, float]:
    """Return (MSE, MAE) as Python floats."""
    mse = torch.mean((pred - target) ** 2).item()
    mae = torch.mean(torch.abs(pred - target)).item()
    return mse, mae


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, criterion: nn.Module) -> tuple[float, float]:
    model.train()
    total_mse, total_mae, n = 0.0, 0.0, 0
    
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        mse, mae = compute_metrics(pred.detach(), y)
        batch = x.size(0)
        total_mse += mse * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)
        mse, mae = compute_metrics(pred, y)
        batch = x.size(0)
        total_mse += mse * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n

## Config and Paths

In [6]:
CSV_PATH = '/kaggle/input/datasets/alaaelmor/ettsmall/ETTh1.csv'
RESULTS_DIR = Path('results')
CKPT_DIR = Path('results/checkpoints')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    'seq_len': 96,       # paper default for iTransformer on ETTh1
    'num_variates': 7,
    'd_model': 64,
    'num_heads': 8,
    'num_layers': 3,
    'dropout': 0.3,
    'lr': 1e-4,
    'batch_size': 32,
    'epochs': 100,
    'patience': 10,
    'seed': SEED,
}

HORIZONS = [96, 192, 336, 720]

# Published targets from Liu et al., ICLR 2024, Table 1 (ETTh1, avg over horizons).
PAPER_AVG_MSE = 0.454
PAPER_AVG_MAE = 0.447

## Training Loop

In [7]:
all_results = []

for pred_len in HORIZONS:
    print(f'\n{"=" * 60}')
    print(f'Horizon: pred_len={pred_len}')
    print(f'{"=" * 60}')

    torch.manual_seed(CONFIG['seed'])
    random.seed(CONFIG['seed'])
    np.random.seed(CONFIG['seed'])

    train_ds = ETTh1Dataset(CSV_PATH, 'train', CONFIG['seq_len'], pred_len)
    val_ds = ETTh1Dataset(CSV_PATH, 'val', CONFIG['seq_len'], pred_len)
    test_ds = ETTh1Dataset(CSV_PATH, 'test', CONFIG['seq_len'], pred_len)

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    
    model = iTransformer(seq_len=CONFIG['seq_len'], pred_len=pred_len, d_model=CONFIG['d_model'], num_heads=CONFIG['num_heads'],
                         num_layers=CONFIG['num_layers'], dropout=CONFIG['dropout']).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {total_params:,}')

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    criterion = nn.MSELoss()

    ckpt_path = str(CKPT_DIR / f'itransformer_ettch1_pred{pred_len}.pt')
    early_stopping = EarlyStopping(patience=CONFIG['patience'], checkpoint_path=ckpt_path)

    log_rows = []
    t0 = time.time()

    for epoch in range(1, CONFIG['epochs'] + 1):
        train_mse, train_mae = train_one_epoch(model, train_loader, optimizer, criterion)
        val_mse, val_mae = evaluate(model, val_loader)
        scheduler.step()

        lr_now = optimizer.param_groups[0]['lr']
        log_rows.append({'epoch': epoch, 'train_mse': round(train_mse, 6), 'train_mae': round(train_mae, 6),
                         'val_mse': round(val_mse, 6), 'val_mae': round(val_mae, 6), 'lr': lr_now})

        if epoch % 10 == 0 or epoch == 1:
            elapsed = time.time() - t0
            print(f'  Epoch {epoch:3d}/{CONFIG["epochs"]} | '
                  f'train MSE {train_mse:.4f} | val MSE {val_mse:.4f} | '
                  f'lr {lr_now:.2e} | {elapsed:.0f}s')

        if early_stopping.step(val_mse, model, epoch):
            print(f'  Early stop at epoch {epoch}. Best epoch: {early_stopping.best_epoch}.')
            break

    log_path = RESULTS_DIR / f'itransformer_ettch1_pred{pred_len}_log.csv'
    pd.DataFrame(log_rows).to_csv(log_path, index=False)

    # Load best checkpoint for test evaluation.
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_mse, test_mae = evaluate(model, test_loader)

    print(f'  Test MSE: {test_mse:.4f} | Test MAE: {test_mae:.4f} | '
          f'Best val MSE: {early_stopping.best_val_mse:.4f} @ epoch {early_stopping.best_epoch}')

    all_results.append({
        'model': 'iTransformer',
        'dataset': 'ETTh1',
        'seq_len': CONFIG['seq_len'],
        'pred_len': pred_len,
        'test_mse': round(test_mse, 6),
        'test_mae': round(test_mae, 6),
        'best_val_mse': round(early_stopping.best_val_mse, 6),
        'best_epoch': early_stopping.best_epoch,
        'num_params': total_params,
        'seed': CONFIG['seed'],
    })

results_df = pd.DataFrame(all_results)
results_path = RESULTS_DIR / 'itransformer_ettch1.csv'
results_df.to_csv(results_path, index=False)
print(f'\nResults saved to {results_path}')
print(results_df[['pred_len', 'test_mse', 'test_mae']].to_string(index=False))


Horizon: pred_len=96
Parameters: 162,528
  Epoch   1/100 | train MSE 0.7101 | val MSE 1.0041 | lr 1.00e-04 | 4s
  Epoch  10/100 | train MSE 0.3471 | val MSE 0.7275 | lr 9.76e-05 | 32s
  Epoch  20/100 | train MSE 0.3248 | val MSE 0.7213 | lr 9.05e-05 | 64s
  Epoch  30/100 | train MSE 0.3113 | val MSE 0.7142 | lr 7.94e-05 | 95s
  Early stop at epoch 34. Best epoch: 24.
  Test MSE: 0.4841 | Test MAE: 0.4831 | Best val MSE: 0.7107 @ epoch 24

Horizon: pred_len=192
Parameters: 168,768
  Epoch   1/100 | train MSE 0.7878 | val MSE 1.2044 | lr 1.00e-04 | 3s
  Epoch  10/100 | train MSE 0.4028 | val MSE 0.9418 | lr 9.76e-05 | 31s
  Epoch  20/100 | train MSE 0.3744 | val MSE 0.9450 | lr 9.05e-05 | 63s
  Early stop at epoch 21. Best epoch: 11.
  Test MSE: 0.5450 | Test MAE: 0.5174 | Best val MSE: 0.9349 @ epoch 11

Horizon: pred_len=336
Parameters: 178,128
  Epoch   1/100 | train MSE 0.8581 | val MSE 1.3661 | lr 1.00e-04 | 3s
  Epoch  10/100 | train MSE 0.4361 | val MSE 1.1746 | lr 9.76e-05 | 31s

## Summary and Benchmark Comparison

In [8]:
avg_mse = results_df['test_mse'].mean()
avg_mae = results_df['test_mae'].mean()

print('=== iTransformer ETTh1 Results ===')
print(results_df[['pred_len', 'test_mse', 'test_mae', 'best_epoch']].to_string(index=False))
print(f'\nAverage | MSE {avg_mse:.4f} | MAE {avg_mae:.4f}')
print(f'Paper   | MSE {PAPER_AVG_MSE:.4f} | MAE {PAPER_AVG_MAE:.4f}')
print(f'Gap     | MSE {avg_mse - PAPER_AVG_MSE:+.4f} | MAE {avg_mae - PAPER_AVG_MAE:+.4f}')

# Expected: iTransformer underperforms PatchTST on ETTh1 (7 variates, strong local temporal
# patterns). This is consistent with the paper -- iTransformer's advantage emerges on
# high-dimensional datasets (ECL: 321 variates, Traffic: 862 variates).

=== iTransformer ETTh1 Results ===
 pred_len  test_mse  test_mae  best_epoch
       96  0.484081  0.483114          24
      192  0.544982  0.517424          11
      336  0.610953  0.564418          24
      720  0.716681  0.628251          31

Average | MSE 0.5892 | MAE 0.5483
Paper   | MSE 0.4540 | MAE 0.4470
Gap     | MSE +0.1352 | MAE +0.1013


## Verify Output Files

In [9]:
required = [RESULTS_DIR / 'itransformer_ettch1.csv'] + [
    RESULTS_DIR / f'itransformer_ettch1_pred{h}_log.csv' for h in HORIZONS
]

all_present = True
for path in required:
    exists = path.exists()
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {path}')
    if not exists:
        all_present = False

if not all_present:
    raise RuntimeError('Some output files are missing. Do not close the session.')
print('\nAll output files verified.')

  [OK] results/itransformer_ettch1.csv
  [OK] results/itransformer_ettch1_pred96_log.csv
  [OK] results/itransformer_ettch1_pred192_log.csv
  [OK] results/itransformer_ettch1_pred336_log.csv
  [OK] results/itransformer_ettch1_pred720_log.csv

All output files verified.
